# 🃏 Poker AI v4 — Multi-Model Kaggle Trainer (HF Dataset Edition)

**Javított verzió:** Az útvonalak és nevek most már pontosan illeszkednek a `train.py` belső logikájához.

### ⚙️ Beállítások:
* **MODEL_CATEGORY:** A mappa neve a HF-en (pl. `2max`).
* **MODEL_NAME_PREFIX:** A modell neve a kiegészítés nélkül (pl. `2max`). A script ebből csinálja majd a `2max_ppo_v4.pth`-t.

In [ ]:
# ==========================================
#        KÖZPONTI KONFIGURÁCIÓ
# ==========================================

MODEL_CATEGORY    = "2max"      # A mappa neve (pl. models/2max)
MODEL_NAME_PREFIX = "2max"      # Ebből lesz: 2max_ppo_v4.pth
EPISODES          = 5000000     # Összes iteráció
UPLOAD_MINUTES    = 30          # Mentési gyakoriság

HF_REPO_ID        = "Mikidikilang/pokerai"
REPO_URL          = "https://github.com/Mikidikilang/PokerAI.git"

# ==========================================

In [ ]:
import os
import subprocess
import torch

CODE_DIR = "/kaggle/working/PokerAI"
MODELS_DIR = f"{CODE_DIR}/models"

print(f"🚀 GPU Elérhető: {torch.cuda.is_available()}")

print("\n📦 GitHub kód letöltése...")
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull"], check=True)

print("🛠️ Függőségek telepítése...")
subprocess.run(["pip", "install", "-r", f"{CODE_DIR}/requirements.txt", "-q"])
subprocess.run(["pip", "install", "huggingface_hub", "-q"])
print("✅ Környezet kész.")

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download
import os

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HuggingFace")
    login(token=hf_token)
    print("✅ Hugging Face Login OK.")
except:
    print("❌ HIBA: 'HuggingFace' secret hiányzik!")
    raise

print(f"\n⬇️ Modell betöltése a Datasetből: {HF_REPO_ID}/models/{MODEL_CATEGORY}...")

try:
    # Letöltjük a kategória mappáját. 
    # A local_dir=CODE_DIR miatt a 'models/2max/...' útvonal jön létre helyben.
    snapshot_download(
        repo_id=HF_REPO_ID,
        repo_type="dataset", 
        allow_patterns=[
            f"models/{MODEL_CATEGORY}/*.pth",
            f"models/{MODEL_CATEGORY}/*.json"
        ],
        ignore_patterns=["*_snap.pth", "*_test.pth"], # Csak a fő modellt szedjük le induláshoz
        local_dir=CODE_DIR,
        local_dir_use_symlinks=False
    )
    
    expected_file = f"{CODE_DIR}/models/{MODEL_CATEGORY}/{MODEL_NAME_PREFIX}_ppo_v4.pth"
    if os.path.exists(expected_file):
        print(f"✅ Modell megtalálva: {expected_file}")
    else:
        print(f"⚠️ A várt fájl nem található ezen az úton: {expected_file}")
        print("📂 Jelenlegi fájlok a mappában:", os.listdir(f"{CODE_DIR}/models/{MODEL_CATEGORY}") if os.path.exists(f"{CODE_DIR}/models/{MODEL_CATEGORY}") else "Mappa nem létezik")
except Exception as e:
    print(f"ℹ️ Hiba vagy nincs előzmény: {e}")

In [ ]:
import threading
import time
import sys
import subprocess
from huggingface_hub import HfApi

api = HfApi()
_stop_sync = [False]
UPLOAD_INTERVAL_SEC = UPLOAD_MINUTES * 60

def _sync_thread():
    while not _stop_sync[0]:
        for _ in range(UPLOAD_INTERVAL_SEC):
            if _stop_sync[0]: break
            time.sleep(1)
            
        if not _stop_sync[0]:
            print(f"\n[HF Sync] ⏳ Automatikus mentés...")
            try:
                api.upload_folder(
                    folder_path=MODELS_DIR,
                    repo_id=HF_REPO_ID,
                    path_in_repo="models",
                    repo_type="dataset",
                    commit_message=f"Kaggle auto-sync: {time.strftime('%H:%M:%S')}"
                )
                print("[HF Sync] ✅ Sikeres feltöltés.")
            except Exception as e:
                print(f"[HF Sync] ❌ Hiba: {e}")

sync_worker = threading.Thread(target=_sync_thread, daemon=True)
sync_worker.start()

# --- TRÉNING INDÍTÁSA ---
# Itt változtattunk: output_dir a CODE_DIR, így megtalálja a /models subfoldert
cmd = [
    sys.executable,
    os.path.join(CODE_DIR, 'train.py'),
    '--model_name', MODEL_NAME_PREFIX,
    '--output_dir', CODE_DIR, 
    '--iterations', str(EPISODES),
]

print('\n' + '=' * 60)
print(f'  🚀 TRÉNING START: {MODEL_NAME_PREFIX}')
print('=' * 60 + '\n')

env = {**os.environ, 'PYTHONPATH': CODE_DIR, 'PYTHONUNBUFFERED': '1'}
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)

try:
    for line in process.stdout:
        print(line, end='', flush=True)
    process.wait()
except KeyboardInterrupt:
    print('\n🛑 Megszakítva. Mentés...')
    process.terminate()
finally:
    _stop_sync[0] = True
    print("\n🚀 Végső mentés...")
    try:
        api.upload_folder(
            folder_path=MODELS_DIR,
            repo_id=HF_REPO_ID,
            path_in_repo="models",
            repo_type="dataset",
            commit_message=f"Final upload after training"
        )
        print("✅ Minden mentve.")
    except Exception as e:
        print(f"❌ Hiba: {e}")